# PostgreSQL Vector Database Retrieval for FinanceRAG

This notebook encodes documents, stores embeddings in PostgreSQL with pgvector, and performs retrieval.

## Prerequisites
1. PostgreSQL with pgvector extension installed
2. `pip install psycopg2-binary`

In [1]:
import json
import logging
import time
import pandas as pd

from financerag.tasks import FinDER
from financerag.retrieval import SentenceTransformerEncoder, PostgresVectorRetrieval
from financerag.tasks.BaseTask import BaseTask

logging.basicConfig(level=logging.INFO)

/opt/homebrew/anaconda3/envs/FinRAG/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/homebrew/anaconda3/envs/FinRAG/lib/python3.11/site-packages/pymilvus/client/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [4]:
# Configuration
CORPUS_PATH = "/Users/vikashpr/Dev/Python/FinanceRAG/icaif-24-finance-rag-challenge/finder_corpus.jsonl/corpus.jsonl"
QUERY_PATH = "/Users/vikashpr/Dev/Python/FinanceRAG/icaif-24-finance-rag-challenge/finder_queries.jsonl/queries.jsonl"
QRELS_PATH = "/Users/vikashpr/Dev/Python/FinanceRAG/icaif-24-finance-rag-challenge/FinDER_qrels.tsv"

# PostgreSQL connection for Postgres.app
# Using host=localhost forces TCP/IP connection instead of Unix socket
POSTGRES_CONNECTION = "postgresql://vikashpr@localhost:5432/financerag"

In [5]:
# Test database connection and pgvector
import psycopg2

try:
    conn = psycopg2.connect(POSTGRES_CONNECTION)
    cur = conn.cursor()
    
    # Check if pgvector extension exists
    cur.execute("SELECT extname FROM pg_extension WHERE extname = 'vector';")
    result = cur.fetchone()
    
    if result:
        print("pgvector extension is installed!")
    else:
        print("pgvector NOT installed. Run this in psql:")
        print("  CREATE EXTENSION vector;")
    
    cur.close()
    conn.close()
    print(f"Connection successful: {POSTGRES_CONNECTION}")
except Exception as e:
    print(f"Connection failed: {e}")
    print("\nTroubleshooting:")
    print("1. Make sure PostgreSQL is running (check Postgres.app)")
    print("2. Try connecting via psql first to verify credentials")

pgvector extension is installed!
Connection successful: postgresql://vikashpr@localhost:5432/financerag


In [6]:
# Load data
def load_jsonl(path):
    with open(path, "r") as f:
        for line in f:
            yield json.loads(line)

corpus = {
    doc["_id"]: {"title": doc.get("title", ""), "text": doc.get("text", "")}
    for doc in load_jsonl(CORPUS_PATH)
}
queries = {q["_id"]: q["text"] for q in load_jsonl(QUERY_PATH)}

df = pd.read_csv(QRELS_PATH, sep="\t")
qrels = df.groupby("query_id").apply(lambda g: dict(zip(g["corpus_id"], g["score"]))).to_dict()

print(f"Corpus: {len(corpus)} documents")
print(f"Queries: {len(queries)}")
print(f"Qrels: {len(qrels)} query judgments")

Corpus: 13863 documents
Queries: 216
Qrels: 64 query judgments


/var/folders/_d/jnx4p5xs4854n467vl8chy_m0000gn/T/ipykernel_83444/724813387.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  qrels = df.groupby("query_id").apply(lambda g: dict(zip(g["corpus_id"], g["score"]))).to_dict()


In [7]:
# Initialize encoder
encoder = SentenceTransformerEncoder(
    model_name_or_path="intfloat/e5-large-v2",
    query_prompt="query: ",
    doc_prompt="passage: ",
)
print("Encoder initialized")

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: intfloat/e5-large-v2


Encoder initialized


In [8]:
# Initialize PostgreSQL Vector Retrieval
retriever = PostgresVectorRetrieval(
    model=encoder,
    connection_string=POSTGRES_CONNECTION,
    table_name="finder_embeddings",
    embedding_dim=1024,  # e5-large-v2 dimension
    batch_size=64,
    index_type="ivfflat",  # Options: 'ivfflat', 'hnsw', 'none'
    recreate_table=True,   # Set False to reuse existing embeddings
)
print("PostgresVectorRetrieval initialized")

INFO:financerag.retrieval.postgres_retrieval:Dropped existing table: finder_embeddings
INFO:financerag.retrieval.postgres_retrieval:Database setup complete. Table: finder_embeddings


PostgresVectorRetrieval initialized


In [ ]:
# Run retrieval (encodes corpus, stores in PostgreSQL, then retrieves)
print("Running retrieval...")
total_start = time.time()

retrieval_results = retriever.retrieve(
    corpus=corpus,
    queries=queries,
    top_k=100,
    score_function="cos_sim"
)

total_time = time.time() - total_start

# Get timing breakdown
timing = retriever.get_timing_metrics()

print(f"\n=== Timing Metrics ===")
print(f"Indexing Time (encode + store): {timing.get('indexing_time', 0):.2f} s")
print(f"Query Encoding Time: {timing.get('query_encoding_time', 0):.2f} s")
print(f"Retrieval Time (DB queries): {timing.get('retrieval_time', 0):.2f} s")
print(f"Avg Time per Query: {timing.get('avg_query_time', 0) * 1000:.2f} ms")
print(f"Total Time: {total_time:.2f} s")

INFO:financerag.retrieval.postgres_retrieval:Encoding and storing 13863 documents...


Running retrieval...


Batches: 100%|██████████| 10/10 [01:56<00:00, 11.65s/it]
INFO:financerag.retrieval.postgres_retrieval:Stored batch 1
Batches: 100%|██████████| 10/10 [02:38<00:00, 15.87s/it]
INFO:financerag.retrieval.postgres_retrieval:Stored batch 2
Batches: 100%|██████████| 10/10 [02:38<00:00, 15.82s/it]
INFO:financerag.retrieval.postgres_retrieval:Stored batch 3
Batches: 100%|██████████| 10/10 [02:31<00:00, 15.16s/it]
INFO:financerag.retrieval.postgres_retrieval:Stored batch 4
Batches: 100%|██████████| 10/10 [02:50<00:00, 17.01s/it]
INFO:financerag.retrieval.postgres_retrieval:Stored batch 5
Batches: 100%|██████████| 10/10 [02:31<00:00, 15.13s/it]
INFO:financerag.retrieval.postgres_retrieval:Stored batch 6
Batches: 100%|██████████| 10/10 [02:15<00:00, 13.51s/it]
INFO:financerag.retrieval.postgres_retrieval:Stored batch 7
Batches: 100%|██████████| 10/10 [02:22<00:00, 14.20s/it]
INFO:financerag.retrieval.postgres_retrieval:Stored batch 8
Batches: 100%|██████████| 10/10 [01:57<00:00, 11.80s/it]
INFO:fi


=== Timing Metrics ===
Indexing Time (encode + store): 1938.05 s
Query Encoding Time: 2.72 s
Retrieval Time (DB queries): 1.31 s
Avg Time per Query: 6.06 ms
Total Time: 1942.11 s


In [10]:
# Evaluate results
ndcg, map_, recall, precision = BaseTask.evaluate(
    qrels=qrels,
    results=retrieval_results,
    k_values=[1, 5, 10],
)

print("\n=== Evaluation Metrics ===")
print(f"NDCG: {ndcg}")
print(f"MAP: {map_}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")

INFO:financerag.tasks.BaseTask:For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
INFO:financerag.tasks.BaseTask:

INFO:financerag.tasks.BaseTask:NDCG@1: 0.3125
INFO:financerag.tasks.BaseTask:NDCG@5: 0.3807
INFO:financerag.tasks.BaseTask:NDCG@10: 0.4247
INFO:financerag.tasks.BaseTask:

INFO:financerag.tasks.BaseTask:MAP@1: 0.2531
INFO:financerag.tasks.BaseTask:MAP@5: 0.3426
INFO:financerag.tasks.BaseTask:MAP@10: 0.3644
INFO:financerag.tasks.BaseTask:

INFO:financerag.tasks.BaseTask:Recall@1: 0.2531
INFO:financerag.tasks.BaseTask:Recall@5: 0.4477
INFO:financerag.tasks.BaseTask:Recall@10: 0.5672
INFO:financerag.tasks.BaseTask:

INFO:financerag.tasks.BaseTask:P@1: 0.3125
INFO:financerag.tasks.BaseTask:P@5: 0.1219
INFO:financerag.tasks.BaseTask:P@10: 0.0797



=== Evaluation Metrics ===
NDCG: {'NDCG@1': 0.3125, 'NDCG@5': 0.38075, 'NDCG@10': 0.42467}
MAP: {'MAP@1': 0.25312, 'MAP@5': 0.34259, 'MAP@10': 0.36437}
Recall: {'Recall@1': 0.25312, 'Recall@5': 0.44766, 'Recall@10': 0.56719}
Precision: {'P@1': 0.3125, 'P@5': 0.12188, 'P@10': 0.07969}


In [12]:
# Summary table
summary = pd.DataFrame({
    'Metric': ['NDCG@1', 'NDCG@5', 'NDCG@10', 'MAP@10', 'Recall@10', 'P@10',
               'Retrieval Time (s)', 'Avg Query Time (ms)'],
    'Value': [
        f"{ndcg['NDCG@1']:.5f}",
        f"{ndcg['NDCG@5']:.5f}",
        f"{ndcg['NDCG@10']:.5f}",
        f"{map_['MAP@10']:.5f}",
        f"{recall['Recall@10']:.5f}",
        f"{precision['P@10']:.5f}",
        f"{timing.get('retrieval_time', 0):.2f}",
        f"{timing.get('avg_query_time', 0) * 1000:.2f}",
    ]
})
print("\n=== Summary ===")
print(summary.to_string(index=False))


=== Summary ===
             Metric   Value
             NDCG@1 0.31250
             NDCG@5 0.38075
            NDCG@10 0.42467
             MAP@10 0.36437
          Recall@10 0.56719
               P@10 0.07969
 Retrieval Time (s)    1.31
Avg Query Time (ms)    6.06
